### Análise do dataset **EM-DAT: The International Disaster Database**

Este dataset contém o registo histórico de desastres naturais globais. Para a nossa investigação, focaremos na frequência e tipologia dos eventos entre **1975 e 2024**. O objetivo é identificar padrões na ocorrência de diferentes tipos de desastres ao longo das décadas.

Citação em linha:
> EM-DAT, CRED / UCLouvain (2025)

Citação completa:
> EM-DAT, CRED / UCLouvain (2025). “The International Disaster Database” [dataset]. Centre for Research on the Epidemiology of Disasters (CRED).

### Classificação dos Tipos de Dados (EM-DAT)

De acordo com a taxonomia de dados, o dataset EM-DAT é composto por variáveis de diferentes naturezas, o que exige diferentes abordagens de visualização:

* **Dados Nominais (Categorias):**
    * `Disaster Type` e `Disaster Subtype`: Identificam a categoria do evento (ex: Flood, Storm). 
    * `Country`, `Region` e `ISO`: Variáveis geográficas que servem para identificação e agrupamento.

* **Dados Temporais:**
    * `Start Year`, `Start Month`, `Start Day`: Representam o momento da ocorrência.
    * `Decade`: Variável derivada (ordinal/temporal) que criámos para reduzir o ruído e observar tendências de longo prazo.

* **Dados Quantitativos (Numéricos):**
    * `Total Deaths`, `Total Affected`: Dados discretos que medem o impacto humano.
    * `Total Damage ('000 US$)`: Dados contínuos que representam o impacto económico.



### Carregamento do Dataset

Carregamos o dataset EM-DAT, que se encontra na mesma pasta que este script. O arquivo é um Excel, e utilizamos a biblioteca `pandas` para ler os dados.


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Carregar o dataset EM-DAT, esta na mesma pasta que o script
file_emdat = '..\data\EMDATdata.xlsx'
df_emdat = pd.read_excel(file_emdat)

df_emdat.head()

FileNotFoundError: [Errno 2] No such file or directory: 'EMDATdata.xlsx'

### Carregamento do Segundo Dataset

Carregamos o dataset das anomalias de temperatura global, que se encontra na mesma pasta que este script. O arquivo é um CSV, e utilizamos a biblioteca `pandas` para ler os dados.


In [ ]:
# Carregar o dataset de anomalia de temperatura
file_temp = 'temperature-anomaly.csv'
df_temp = pd.read_csv(file_temp)


df_temp.head()

,Entity,Code,Year,Average,Lower bound,Upper bound
0,Northern Hemisphere,OWID_NH,1850,-0.055067,-0.254982,0.144848
1,Northern Hemisphere,OWID_NH,1851,0.161580,-0.047569,0.370729
2,Northern Hemisphere,OWID_NH,1852,0.145127,-0.076860,0.367114
3,Northern Hemisphere,OWID_NH,1853,0.135437,-0.082370,0.353244
4,Northern Hemisphere,OWID_NH,1854,0.206140,0.009939,0.402340


1: A Escala do Problema - O Aumento da Frequência de Desastres

Para iniciar o nosso relatório, analisamos a evolução do número total de desastres naturais registados anualmente entre 1975 e 2024. Este gráfico serve para estabelecer o contexto de urgência, demonstrando como a ocorrência de eventos extremos se tornou mais comum nas últimas décadas.

Seguindo os princípios da disciplina:
* **Tempo no Eixo X:** Como recomendado na Aula 4, utilizamos o eixo horizontal para a progressão temporal.
* **Data-Ink Ratio:** Removemos elementos desnecessários e focamos na linha de tendência para facilitar a leitura.
* **Título de Insight:** O título comunica imediatamente a descoberta principal.

In [ ]:

# 1. Agrupar os dados para contar o número de desastres por ano
# Usamos o 'DisNo.' como identificador único para a contagem
frequencia_anual = df_emdat.groupby('Start Year').size().reset_index(name='Total de Desastres')

# 2. Criar o gráfico de linha para mostrar a tendência ao longo do tempo
fig = px.line(
    frequencia_anual, 
    x='Start Year', 
    y='Total de Desastres',
    title="<b>A Frequência de Desastres Naturais Triplicou desde 1975</b><br><span style='font-size:14px; color:gray'>Número total de eventos registados por ano (Global)</span>",
    labels={'Start Year': 'Ano', 'Total de Desastres': 'Nº de Ocorrências'},
    markers=True # Adiciona pontos para facilitar a leitura de anos específicos
)

# 3. Formatação do gráfico 
fig.update_traces(line_color='darkred', line_width=3) # Cor de destaque

fig.update_layout(
    plot_bgcolor='white',
    hovermode='x unified', 
    xaxis=dict(showgrid=True, gridcolor='whitesmoke'),
    yaxis=dict(showgrid=True, gridcolor='whitesmoke')
)

# Adicionar anotação 
ano_pico = frequencia_anual.loc[frequencia_anual['Total de Desastres'].idxmax(), 'Start Year']
valor_pico = frequencia_anual['Total de Desastres'].max()

fig.add_annotation(
    x=ano_pico, y=valor_pico,
    text=f"Pico histórico em {int(ano_pico)}",
    showarrow=True, arrowhead=2,
    ax=0, ay=-40,
    font=dict(color="darkred", size=12)
)

fig.show()

## Reflexão: 

Ao analisar a tendência de crescimento no gráfico anterior, é impossível ignorar que o número de desastres registados anualmente **triplicou** entre 1975 e 2024. No entanto devemos adotar uma postura crítica (EDA) e questionar: *O que explica esta aceleração?*.


### 2: A Impressão Digital das Alterações Climáticas

Nesta análise, dividimos os desastres por tipo. A hipótese é que o aumento observado é impulsionado quase exclusivamente por eventos **meteorológicos, hidrológicos e climatológicos** (como inundações e tempestades), enquanto eventos **geofísicos** (terramotos), que não dependem do clima, permanecem estáveis ao longo dos 50 anos.

Princípios aplicados:
* **Gráfico de Áreas Empilhadas:** Ideal para mostrar como a composição de um "todo" muda ao longo do tempo.
* **Cores Categóricas:** Utilizamos uma paleta distinta para cada tipo de desastre.

In [ ]:
# 1. Preparar os dados: Contar por Ano e por Tipo de Desastre
# Filtramos apenas o essencial para evitar ruído visual (Data-Ink Ratio)
df_composition = df_emdat[(df_emdat['Start Year'] >= 1975) & (df_emdat['Start Year'] <= 2024)]
df_composition = df_composition.groupby(['Start Year', 'Disaster Type']).size().reset_index(name='Count')

# 2. Criar o Gráfico de Áreas Empilhadas (Stacked Area Chart)
fig = px.area(
    df_composition, 
    x="Start Year", 
    y="Count", 
    color="Disaster Type",
    title="<b>Inundações e Tempestades Dominam o Aumento de Desastres</b><br><span style='font-size:14px; color:gray'>Evolução da tipologia de eventos extremos (1975-2024)</span>",
    labels={'Start Year': 'Ano', 'Count': 'Nº de Desastres', 'Disaster Type': 'Tipo'},
    color_discrete_sequence=px.colors.qualitative.Prism # Paleta categórica clara
)

# 3. Estilização Editorial
fig.update_layout(
    plot_bgcolor='white',
    hovermode='x unified',
    xaxis=dict(showgrid=True, gridcolor='whitesmoke'),
    yaxis=dict(showgrid=True, gridcolor='whitesmoke'),
    legend_title="Tipo de Desastre"
)

fig.show()

### Reflexão:

### 3: O Impacto - Onde a Crise se Intensifica?

 Utilizamos um **Heatmap** para identificar "hotspots" de desastres, cruzando as regiões do globo com a linha temporal.

De acordo com os princípios da disciplina:
* **Visualização de Densidade:** O heatmap é ideal para mostrar padrões de concentração em grandes volumes de dados.
* **Data-Ink Ratio:** Simplificamos a grelha para que a variação de cor (a "temperatura" do gráfico) seja o foco principal.
* **Título de Insight:** O título orienta o leitor para a região com maior aceleração de eventos.

In [ ]:
# 1. Preparar os dados: Contar desastres por Ano e Região
# Filtramos os anos (1975-2024) e removemos possíveis valores nulos em 'Region'
df_heatmap = df_emdat[(df_emdat['Start Year'] >= 1975) & (df_emdat['Start Year'] <= 2024)].copy()
heatmap_data = df_heatmap.groupby(['Start Year', 'Region']).size().reset_index(name='Frequência')

# 2. Criar o Heatmap
# Usamos density_heatmap ou imshow para representar a densidade
fig = px.density_heatmap(
    heatmap_data, 
    x="Start Year", 
    y="Region", 
    z="Frequência",
    title="<b>Hotspots Globais: A Intensificação de Desastres por Região</b><br><span style='font-size:14px; color:gray'>Frequência anual de eventos extremos por zona geográfica (1975-2024)</span>",
    labels={'Start Year': 'Ano', 'Region': 'Região', 'Frequência': 'Nº de Eventos'},
    color_continuous_scale='YlOrRd', # Escala sequencial de amarelo a vermelho (perigo/calor)
    nbinsx=50 # Garante que cada ano tem a sua coluna
)

fig.update_layout(
    plot_bgcolor='white',
    xaxis_title="Evolução Temporal (Anos)",
    yaxis_title="Regiões do Globo",
    coloraxis_colorbar=dict(title="Nº Eventos")
)

fig.show()

Causa e Efeito — Temperatura vs. Frequência de Desastres

Nesta etapa, cruzamos a "assinatura térmica" do planeta com a ocorrência de eventos extremos. Embora os dados venham de fontes distintas, utilizamos um eixo temporal comum para observar se os picos de aquecimento global coincidem com o aumento na atividade de desastres.

De acordo com as diretrizes de design (Aula 4 e 5):
* **Painéis Empilhados:** Em vez de usar dois eixos Y no mesmo gráfico (o que dificultaria a leitura), utilizamos subplots para manter as unidades (°C e contagem) separadas e claras.
* **Sincronização Temporal:** O tempo ocupa sempre o eixo X, garantindo a continuidade da história iniciada no Ato 1.

In [ ]:
from plotly.subplots import make_subplots

# 1. Carregar e preparar o dataset de temperatura
df_temp = pd.read_csv('temperature-anomaly.csv')
df_temp_world = df_temp[(df_temp['Entity'] == 'World') & (df_temp['Year'] >= 1975) & (df_temp['Year'] <= 2024)]

# 2. Preparar a contagem de desastres por ano 
disaster_counts = df_emdat[(df_emdat['Start Year'] >= 1975) & (df_emdat['Start Year'] <= 2024)]
disaster_counts = disaster_counts.groupby('Start Year').size().reset_index(name='Count')

# 3. Criar Subplots (2 linhas, 1 coluna)
fig = make_subplots(rows=2, cols=1, 
                    shared_xaxes=True, 
                    vertical_spacing=0.1,
                    subplot_titles=("Anomalia de Temperatura Global (°C)", "Número Total de Desastres Naturais"))

# Adicionar Temperatura (Causa) - Linha Vermelha
fig.add_trace(
    go.Scatter(x=df_temp_world['Year'], y=df_temp_world['Average'], 
               name="Temperatura", line=dict(color='#d62728', width=2.5)),
    row=1, col=1
)

# Adicionar Desastres (Efeito) - Linha Azul
fig.add_trace(
    go.Scatter(x=disaster_counts['Start Year'], y=disaster_counts['Count'], 
               name="Nº Desastres", line=dict(color='#1f77b4', width=2.5)),
    row=2, col=1
)

# 4. Ajustar Layout 
fig.update_layout(
    title_text="<b>O Aquecimento Global como Catalisador de Eventos Extremos</b><br><span style='font-size:14px; color:gray'>Comparação entre anomalias térmicas e frequência de desastres (1975-2024)</span>",
    height=650,
    plot_bgcolor='white',
    showlegend=False
)

# Limpeza de grelhas e eixos
fig.update_xaxes(showgrid=True, gridcolor='#f0f0f0')
fig.update_yaxes(showgrid=True, gridcolor='#f0f0f0')

fig.show()

### O Impacto Económico — A Explosão dos Custos

Atendendo mais a research question> How have the frequency and economic damage of extreme weather events changed
over the past 50 years, and which world regions show the steepest acceleration?

Focamo-nos na dimensão económica da nossa pergunta de investigação. Este gráfico utiliza a variável `Total Damage, Adjusted ('000 US$)`, que representa o valor das perdas financeiras ajustadas pela inflação para garantir uma comparação justa entre 1975 e 2024.

In [ ]:

# 1. Limpeza: Filtrar registos com dados de danos e anos do estudo
df_damage = df_emdat[df_emdat["Total Damage, Adjusted ('000 US$)"].notnull()].copy()
df_damage = df_damage[(df_damage['Start Year'] >= 1975) & (df_damage['Start Year'] <= 2024)]

# 2. Agrupar por Ano e Tipo de Desastre para ver a composição do custo
damage_trend = df_damage.groupby(['Start Year', 'Disaster Type'])["Total Damage, Adjusted ('000 US$)"].sum().reset_index()

# 3. Criar o Gráfico de Áreas Empilhadas
fig = px.area(
    damage_trend, 
    x="Start Year", 
    y="Total Damage, Adjusted ('000 US$)", 
    color="Disaster Type",
    title="<b>Danos Económicos: O Custo dos Eventos Extremos disparou no Séc. XXI</b><br><span style='font-size:14px; color:gray'>Total anual de danos ajustados em milhares de USD (1975-2024)</span>",
    labels={'Start Year': 'Ano', "Total Damage, Adjusted ('000 US$)": 'Danos (kUSD)'},
    color_discrete_sequence=px.colors.qualitative.Vivid
)

# 4. Estilo Editorial (Aula 4/5)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='whitesmoke'),
    yaxis=dict(showgrid=True, gridcolor='whitesmoke'),
    legend_title="Tipo de Desastre"
)

fig.show()